# Lab 04: LLM Routing

Not every question needs your best model. "What's your return policy?" runs fine on a small, cheap model; a multi-step troubleshooting question is worth the expensive one. Routing is just sending each query to the right model. The whole game is deciding which is which.

We'll let a cheap LLM make that call: it tags the query simple or complex before we answer, which catches wording a keyword match would miss.

**What you'll do:**
- Classify query complexity with a small model
- Route simple queries to Haiku, complex ones to Sonnet
- Check the routing decisions in Langfuse
- See where the cost actually moves (it's not where you'd expect)

**Prerequisites:** Labs 01–03.

```
01 Baseline → 02 Quick Wins → 03 Caching → [04 Routing] → 05 Guardrails → 06 Gateway → 07 Evaluations
                                               ↑ You are here
```

## Step 1: Setup

In [1]:
from __future__ import annotations

import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_runtime = Runtime()

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'Not set')}")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


## Step 2: Why route between models?

A "what's your return policy?" doesn't need Sonnet's reasoning. Haiku answers it just as well for a fraction of the price.

- **Haiku** — about 3x cheaper than Sonnet per token. Good for Q&A, lookups, greetings. Needs a 4,096-token prompt before caching kicks in.
- **Sonnet** — the stronger model, for reasoning and troubleshooting. Caches from 1,024 tokens up.

Our system prompt is ~1,030 tokens, which clears Sonnet's caching bar but not Haiku's. So the queries we route to Haiku get no caching discount — a wrinkle we come back to once we look at the real cost math.

**Three ways to decide where a query goes:**

- **Keyword matching** — fast, free, but brittle. Misses anything phrased differently than you expected.
- **LLM classification** — handles odd phrasing and edge cases, but costs an extra model call before every answer.
- **Embedding similarity** — no LLM call, but you need training data and more moving parts.

We'll use LLM classification with Haiku here, because it's the easiest to follow in a notebook. The next cell shows where it sits among the other options and what it costs you.

## The routing design space

Keyword, LLM-classifier, and embedding are three points on a bigger map. A 2025 survey lays out six routing patterns and a common shape they all share — a pre-router, a quality estimator, and an escalation policy — that you mix and match depending on your constraints ([Dynamic Model Routing and Cascading: A Survey, 2025](https://arxiv.org/abs/2603.04445)). Here are the ones you'll actually reach for:

| Pattern | How it decides | When to use | Decision latency |
|---|---|---|---|
| **Rule / keyword** | Static `if/else` on length, intent tags, tool name | Tasks already tagged upstream (each agent tool maps to a tier) | ~0ms |
| **Embedding classifier** | Embed query, score against learned clusters | High volume, clear difficulty boundary, latency-sensitive chat | a vector lookup (no LLM call) |
| **LLM-as-router classifier** *(this lab)* | A cheap LLM emits a complexity label before answering | Bimodal workloads (simple FAQ + complex analysis); fastest to prototype | a full extra LLM call in the critical path |
| **Learned preference router** | Router trained on human preference data (RouteLLM) | Production at scale; best cost-quality Pareto frontier | a trained model, negligible vs generation |
| **Cascade / confidence escalation** | Cheap model answers first; escalate only if a verifier is unsure | Verifiable outputs (code→tests, extraction→schema); no clean upfront difficulty signal | worst case = sum of all hops |

We picked the LLM-as-router classifier because it's the clearest one to demo. Just know it isn't the cheapest at runtime — it adds a whole LLM call before every query. The patterns lower in the table avoid that.

### What the research actually found

- **Routing pays off because traffic is lopsided.** FrugalGPT (Chen, Zaharia, Zou 2023) and RouteLLM (Ong et al., 2024) both found **40–70% of production queries** can go to the cheap model with no noticeable quality drop.
- **Trained routers beat LLM-classifiers on cost.** RouteLLM's preference-trained routers hit **95% of strong-model quality while cutting strong-model calls by up to ~85%** on MT-Bench, with **under 1% routing overhead** — compare that to the full extra call our classifier makes on *every* request ([RouteLLM, Ong et al. 2024](https://arxiv.org/abs/2406.18665); [LMSYS blog](https://lmsys.org/blog/2024-07-01-routellm/)).
- **Routing up front vs. cascading is a genuine tradeoff.** Deciding before you answer (what we do) saves a wasted generation but can misclassify. Cascading is usually more accurate but eats full tail latency when the cheap answer gets rejected ([SATER, EMNLP 2025](https://aclanthology.org/2025.emnlp-main.531.pdf); [cascade routing, ICML 2025](https://proceedings.mlr.press/v267/dekoninck25a.html)).
- **One signal isn't enough.** Token count is a poor proxy for difficulty; combining task type, complexity, and cost beats any single signal. Task type is the most useful one, because different tasks lean on the model differently — code and math need capability, summarization and translation don't.

### Caching flips the cost math

That ~3x Haiku savings from Step 2 shrinks fast once caching is in play. A *cached* Sonnet input read is so much cheaper than a full-price token that a cached Sonnet request can come in *under* an uncached Haiku one. And since our prompt clears Sonnet's caching floor but not Haiku's, the queries we send to Haiku get no discount at all. The lesson: weigh cache hit-rate when you route, don't assume "smaller model" always means "cheaper request." (Current per-model and cached-read rates are on the [Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/).)

### What this lab skips (but production shouldn't)

- **Pick a quality floor first, then cut cost under it.** "Save money" isn't a target; "stay above this quality bar per task type, as cheaply as possible" is.
- **Routers drift — shadow-test them.** Run a new policy on real traffic but only *log* its decisions for a week, then compare quality before you flip it on. A routing change is a silent quality change.
- **Calibration matters.** A confidence-gated cascade only works if the cheap model knows when it's unsure. Overconfident small models break it.
- **Fallback isn't retry.** A 400 fails the same way on the backup model; a 503 should reroute; a 429 needs backoff. Our classifier just defaults to Sonnet on any failure — safe, slightly pricey, and not the nuanced handling you'd ship.

> **For production:** start with static task-based routing (free, no training data), add caching and a semantic cache, and only build a learned or classifier router once the monthly bill justifies it.

*Sources: [RouteLLM (Ong et al. 2024)](https://arxiv.org/abs/2406.18665) · [Routing & Cascading Survey (2025)](https://arxiv.org/abs/2603.04445) · [Cascade Routing, ICML 2025](https://proceedings.mlr.press/v267/dekoninck25a.html) · [SATER, EMNLP 2025](https://aclanthology.org/2025.emnlp-main.531.pdf) · [LMSYS RouteLLM blog](https://lmsys.org/blog/2024-07-01-routellm/)*

### Query Classification Examples

These examples match our test prompts:

**Simple → Haiku:**
- "What is your return policy for laptops?" — Single factual lookup
- "Tell me about your smartphone options" — Direct product question
- "Hello! What can you help me with today?" — Greeting

**Complex → Sonnet:**
- "My laptop won't turn on, can you help me troubleshoot?" — Multi-step troubleshooting
- "I want to buy a laptop. What are the specs and what's the return policy?" — Multiple questions

## Step 3: Review the Routing Logic

The v4 agent uses Haiku to classify queries before routing to the appropriate model.

In [2]:
from agents.v4_routing import CLASSIFIER_PROMPT

print("=== CLASSIFIER PROMPT ===")
print(CLASSIFIER_PROMPT)

=== CLASSIFIER PROMPT ===
Classify customer support queries for model routing.

SIMPLE queries (route to cheaper model):
- Greetings and general questions
- Single factual lookups (price, policy, hours)
- Direct questions with straightforward answers

COMPLEX queries (route to powerful model):
- Multi-step troubleshooting
- Product comparisons or recommendations
- Questions requiring analysis or reasoning
- Multiple questions in one message

Respond with ONLY one word: simple or complex

Examples:

Query: "What is your return policy for laptops?"
simple

Query: "Tell me about your smartphone options"
simple

Query: "Hello! What can you help me with today?"
simple

Query: "My laptop won't turn on, can you help me troubleshoot?"
complex

Query: "I want to buy a laptop. What are the specs and what's the return policy?"
complex


### How the classifier works

1. Haiku gets the query plus the classifier prompt.
2. It answers with one word, "simple" or "complex" (`max_tokens=50`, `temperature=0`).
3. The router checks that word and picks the model: "simple" → Haiku answers; anything else, including any error, → Sonnet answers.

That's one extra Haiku call per query: the classifier prompt and the query in, a single word out. Cheap next to a Sonnet answer, but not free, and it sits in front of *every* request. (The patterns cell covers why a learned or embedding router skips this cost.)

> **Honest note for the demo.** Our 5 test prompts are the same examples written into the classifier prompt, so the 3-simple/2-complex split below is rigged — the classifier can't misroute prompts it's already memorized. On real traffic, published routers see **40–70%** of queries go to the cheap model (FrugalGPT, RouteLLM), but that depends entirely on your workload, and a real router *will* get some wrong. Measure your own split with shadow traffic before trusting any savings number.

In [3]:
# Review the v4 agent code
agent_file = Path("agents/v4_routing.py")
print(agent_file.read_text())

"""
V4 Routing Agent - Model routing based on query complexity.
- Same system prompt as v3 with prompt caching
- LLM-based classification using Haiku
- Simple queries -> Haiku (cheaper)
- Complex queries -> Sonnet (better quality)
"""

from __future__ import annotations

import json
import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from opentelemetry import trace
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands.types.content import SystemContentBlock

from utils.agent_config import MODEL_HAIKU, MODEL_SONNET, SYSTEM_PROMPT_TEXT, setup_langfuse_telemetry
from utils.tools import get_product_info, get_return_policy, get_technical_support, web_search

setup_langfuse_telemetry()

app = BedrockAgentCoreApp()

# Classifier prompt for LLM-based routing
CLASSIFIER_PROMPT = """Classify customer support queries for model routing.

SIMPLE queries (route to cheaper model):
- Greetings and general question

## Step 4: Deploy the Routing Agent

In [4]:
agent_name = "customer_support_v4_routing"
agent_file = str(Path("agents/v4_routing.py").absolute())
requirements_file = str(Path("requirements-for-agentcore.txt").absolute())

print(f"Agent name: {agent_name}")
print(f"Agent file: {agent_file}")
print(f"Requirements: {requirements_file}")

# Clean up any existing build files from previous labs
for f in ["Dockerfile", ".dockerignore", ".bedrock_agentcore.yaml"]:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed existing: {f}")

print(f"Configuring agent: {agent_name}")
agentcore_runtime.configure(
    entrypoint=agent_file,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=requirements_file,
    region=region,
    agent_name=agent_name,
)

Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/agents/v4_routing.py, bedrock_agentcore_name=v4_routing
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_v4_routing


Agent name: customer_support_v4_routing
Agent file: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/agents/v4_routing.py
Requirements: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/requirements-for-agentcore.txt
Removed existing: Dockerfile
Removed existing: .dockerignore
Removed existing: .bedrock_agentcore.yaml
Configuring agent: customer_support_v4_routing


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.dockerignore
Setting 'customer_support_v4_routing' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/Dockerfile'), dockerignore_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='739907928487', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

In [5]:
# Modify Dockerfile for Langfuse
dockerfile_path = Path("Dockerfile")
if dockerfile_path.exists():
    content = dockerfile_path.read_text()
    if "opentelemetry-instrument" in content:
        import re

        content = re.sub(
            r'CMD \["opentelemetry-instrument", "python", "-m", "([^"]+)"\]', r'CMD ["python", "-m", "\1"]', content
        )
        dockerfile_path.write_text(content)
        print("Dockerfile modified for Langfuse")
    else:
        print("Dockerfile already configured or using different format")
else:
    print("Dockerfile not found - will be created during deployment")

Dockerfile modified for Langfuse


In [6]:
env_vars = {
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL"),
    "LANGFUSE_PUBLIC_KEY": os.environ.get("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.environ.get("LANGFUSE_SECRET_KEY"),
    "PYTHONUNBUFFERED": "1",
}

print("Deploying to AgentCore Runtime...")
launch_result = agentcore_runtime.launch(env_vars=env_vars, auto_update_on_conflict=True)
agent_arn = launch_result.agent_arn
print(f"Agent deployed: {agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_v4_routing' to account 739907928487 (us-east-1)
Generated image tag: 20260602-172218-747
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_v4_routing


Deploying to AgentCore Runtime...


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v4_routing
Getting or creating execution role for agent: customer_support_v4_routing
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-0fb396fd48


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v4_routing


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-0fb396fd48
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-0fb396fd48
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: customer_support_v4_routing
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-0fb396fd48
Reusing existing CodeBuild execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-0fb396fd48
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: customer_support_v4_routing/source.zip
Updated CodeBuild project: bedrock-agentcore-customer_support_v4_routing-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.6s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 6.4s
🔄 BUILD start

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v4_routing-3LNaMtHxM3


In [7]:
# Save the agent ARN for later use
agent_arn = launch_result.agent_arn
print(f"Agent ARN: {agent_arn}")

# Grant the auto-created execution role the shared runtime permissions
# (SSM KB lookup + Bedrock Retrieve + ApplyGuardrail) used across all labs.
from utils.runtime_helpers import ensure_runtime_permissions

ensure_runtime_permissions(region)

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v4_routing-3LNaMtHxM3
Granted customer-support runtime permissions to AmazonBedrockAgentCoreSDKRuntime-us-east-1-0fb396fd48


## Step 5: Test Model Routing

Let's run the same test prompts and observe which model handles each query.

In [8]:
def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))

In [9]:
from utils.langfuse_metrics import (
    clear_metrics,
    collect_metric,
    get_latest_trace_metrics,
    print_metrics,
    print_metrics_table,
)

clear_metrics()

TEST_PROMPTS = [
    ("Return Policy", "What is your return policy for laptops?"),
    ("Product Info", "Tell me about your smartphone options"),
    ("Technical Support", "My laptop won't turn on, can you help me troubleshoot?"),
    ("Multi-part Question", "I want to buy a laptop. What are the specs and what's the return policy?"),
    ("General Question", "Hello! What can you help me with today?"),
]

for test_name, prompt in TEST_PROMPTS:
    print("=" * 60)
    print(f"Test: {test_name}")
    print("=" * 60)

    result = invoke_agent(prompt)

    if isinstance(result, dict):
        print(f"Model used: {result.get('model_used', 'N/A')}")
        print(f"Complexity: {result.get('complexity', 'N/A')}")
        print(f"Response: {str(result.get('response', result))[:200]}...")
    else:
        print(result)

    # Get metrics from parent trace (includes classifier + main agent)
    metrics = get_latest_trace_metrics(
        agent_name="customer-support-v4-routing",
        wait_seconds=5,
        max_retries=5,
        timeout_seconds=120,
    )
    print_metrics(metrics, test_name)
    collect_metric(metrics, test_name)

Test: Return Policy
Model used: us.anthropic.claude-haiku-4-5-20251001-v1:0
Complexity: simple
Response: - **answer:** Here's our return policy for laptops:

  **Return Window:** 30 days from purchase

  **Condition Requirements:**
  - Original packaging and all accessories included
  - No physical damag...

                    LANGFUSE METRICS
  Test:          Return Policy
  Trace ID:      0a7e7268537f5a1ba9915880dd0014ca
------------------------------------------------------------
  Latency:       4.93s
  Cost:          $0.008154
  Input tokens:  6,749
  Output tokens: 281
  Total tokens:  7,030
------------------------------------------------------------
  Cache read tokens:   0
  Cache write tokens:  0
------------------------------------------------------------
  View trace:    https://d1fnzg75c19u2d.cloudfront.net/trace/0a7e7268537f5a1ba9915880dd0014ca

Test: Product Info
Model used: us.anthropic.claude-haiku-4-5-20251001-v1:0
Complexity: simple
Response: - **answer:** Here's ou

In [10]:
print_metrics_table()

# Save metrics for comparison in later notebooks
from utils.langfuse_metrics import save_metrics

save_metrics("v4")


                                  METRICS SUMMARY
               Test Latency    Cost Input Output Cache Read Tokens Cache Write Tokens
      Return Policy   4.93s $0.0082 6,749    281                 0                  0
       Product Info   6.02s $0.0086 6,793    354                 0                  0
  Technical Support  10.28s $0.0099 1,518    275             5,678                  0
Multi-part Question  10.51s $0.0143 1,444    583             5,678                  0
   General Question   2.86s $0.0040 3,381    125                 0                  0
---------------------------------------------------------------------------------------------------------
  TOTALS: Latency(avg): 6.92s | Cost: $0.0450 | Input: 19,885 | Output: 1,618
          Cache Read Tokens: 11,356 | Cache Write Tokens: 0

Metrics saved as 'v4' → .lab_metrics.json


### What just routed where

**To Haiku (3):** Return Policy (lookup), Product Info (direct question), General Question (greeting).

**To Sonnet (2):** Technical Support (multi-step), Multi-part Question (several questions at once).

3 of 5 went to the cheap model. Remember this split is rigged for the demo — the prompts are the classifier's own examples — so read it as "here's the mechanism working," not a production hit rate.

## Step 6: Compare with v3 (Caching)

Enter your metrics from Lab 03 (v3 caching) to compare with v4 routing results.

In [11]:
from utils.langfuse_metrics import load_metrics, print_comparison

# Load metrics from Lab 03 (saved automatically when you ran print_metrics_table())
v3 = load_metrics("v3")

# Or enter manually if Lab 03 metrics weren't saved:
# v3 = {"total_cost": 0.0438, "avg_latency": 8.10, "total_input_tokens": 4228, "total_output_tokens": 1795}

# Print comparison (current metrics auto-calculated from collected)
print_comparison(
    prev_name="v3 (Caching)",
    curr_name="v4 (Routing)",
    prev_cost=v3["total_cost"],
    prev_latency=v3["avg_latency"],
    prev_input_tokens=v3["total_input_tokens"],
    prev_output_tokens=v3["total_output_tokens"],
)

Loaded 'v3' metrics: cost=$0.0463, latency=7.30s, input=4,632, output=1,705
  V3 (CACHING) vs V4 (ROUTING) COMPARISON
Metric                     v3 (Caching)       v4 (Routing)       Change
----------------------------------------------------------------------
Total Cost           $           0.0463 $           0.0450        -2.8%
Avg Latency (s)                    7.30               6.92        -5.3%
Input Tokens                      4,632             19,885      +329.3%
Output Tokens                     1,705              1,618        -5.1%

Result: 2.8% cost reduction, 5.3% latency improvement


## Summary

We routed queries by difficulty: a Haiku classifier tags each one simple or complex, simple goes to Haiku, complex to Sonnet.

### Why the cost barely moved vs v3

The small cost change *is* the lesson here. Three things blunt the savings, all visible in the metrics above:

- **Caching works against the route.** The Haiku rows show full-price input tokens and zero cache reads; the Sonnet rows show a tiny input plus a big cached read. Our prompt clears Sonnet's 1,024-token caching floor but not Haiku's 4,096 — so the "cheap" requests are uncached and the "expensive" ones are discounted. A cached Sonnet call can cost about the same as an uncached Haiku one.
- **The classifier isn't free** — every request pays for an extra Haiku call just to pick a model.
- **v3 was already cheap** — caching took most of the headroom.

### What to take away

- **Route and cache together.** "Smaller model" isn't automatically "cheaper request" — weigh cache hit-rate per tier before assuming a route saves money.
- **LLM classification catches phrasing keywords miss**, but adds an LLM call to every request. A learned or embedding router does the same job for far less overhead (see the patterns cell).
- **Failures fall back to Sonnet** — safe, slightly pricier, not a guarantee the routing was right.

**Next:** Lab 05 adds Bedrock Guardrails to catch off-topic queries before they hit the model.

---

## Cleanup

To delete the agent deployed in this notebook, uncomment and run the following code.

In [12]:
# # Uncomment to delete resources created in this lab
# agentcore_runtime.destroy(delete_ecr_repo=True)
# print(f"Deleted agent and ECR repository: {agent_name}")